# Витрина обращений NYC 311

ЛР № 4, переход из staging в mart

Выбран основной вопрос Project Brief

> Как различались число обращений и медианное время до закрытия по типам проблем и ответственным агентствам в Манхэттене в январе 2025 года?

Для будущих расчётов нужны число обращений и время до закрытия в часах, срезы по `complaint_type` и `agency`.
В этой работе собирается витрина и проверяется её пригодность, полный расчёт показателей остаётся следующим этапам

ЛР № 3 исключила 27 записей с противоречивыми датами.
Обращения без даты закрытия остались в staging с NULL, их нельзя исключать из числа обращений или считать время закрытия нулевым

In [1]:
from pathlib import Path

STAGING_PATH = "s3://staging/nyc311_manhattan/source_ingested_on=2026-10-14/staging.parquet"
OUTPUT_PREFIX = "s3://mart/nyc311_manhattan/source_ingested_on=2026-10-14"
MART_PATH = f"{OUTPUT_PREFIX}/service_requests.parquet"
PERIOD_START = "2025-01-01"
PERIOD_END = "2025-02-01"

assert STAGING_PATH.rsplit("/", 2)[-2] == OUTPUT_PREFIX.rsplit("/", 1)[-1]
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## Подключение и входные данные

Подключение к S3, загрузка `.env` и вывод небольших таблиц переиспользованы из ЛР № 3.
Данные читаются DuckDB непосредственно из S3, ключи доступа не выводятся

In [2]:
import hashlib
import os
from html import escape
from urllib.parse import urlsplit

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(PROJECT_ROOT / ".env")
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs FROM 'https://extensions.duckdb.org'")
con.execute("LOAD httpfs")
settings = {
    "s3_endpoint": os.environ["S3_ENDPOINT"],
    "s3_access_key_id": os.environ["S3_ACCESS_KEY"],
    "s3_secret_access_key": os.environ["S3_SECRET_KEY"],
    "s3_region": "us-east-1",
    "s3_url_style": "path",
    "s3_use_ssl": False,
}
for name, value in settings.items():
    con.execute(f"SET {name} = ?", [value])
s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{os.environ['S3_ENDPOINT']}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)

In [3]:
def show_rows(columns, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_query(sql, params=None):
    result = con.execute(sql, params or [])
    rows = result.fetchall()
    show_rows([column[0] for column in result.description], rows)
    return rows


def fingerprint(s3_path):
    parsed = urlsplit(s3_path)
    response = s3.get_object(Bucket=parsed.netloc, Key=parsed.path.lstrip("/"))
    with response["Body"] as body:
        digest = hashlib.sha256(body.read()).hexdigest()
    return {
        "sha256": digest,
        "bytes": response["ContentLength"],
        "etag": response["ETag"],
        "last_modified": response["LastModified"].isoformat(),
    }


input_before = fingerprint(STAGING_PATH)
con.read_parquet(STAGING_PATH, hive_partitioning=False).create_view("input_staging")
staging_rows = con.execute("SELECT count(*) FROM input_staging").fetchone()[0]
input_schema = {row[0]: row[1] for row in con.execute("DESCRIBE input_staging").fetchall()}
required_input = {
    "unique_key": "VARCHAR",
    "created_date": "TIMESTAMP",
    "closed_date": "TIMESTAMP",
    "complaint_type": "VARCHAR",
    "agency": "VARCHAR",
    "quality_missing_complaint_type": "BOOLEAN",
    "quality_missing_agency": "BOOLEAN",
}
assert all(input_schema.get(field) == kind for field, kind in required_input.items())
show_rows(["Вход", "Строки", "Поля"], [[STAGING_PATH, staging_rows, len(input_schema)]])
show_query("DESCRIBE input_staging")
show_query("""
    SELECT unique_key, created_date, closed_date, complaint_type, agency,
        quality_missing_complaint_type, quality_missing_agency
    FROM input_staging ORDER BY unique_key LIMIT 3
""")

Вход,Строки,Поля
s3://staging/nyc311_manhattan/source_ingested_on=2026-10-14/staging.parquet,61304,10


column_name,column_type,null,key,default,extra
unique_key,VARCHAR,YES,None,None,None
created_date,TIMESTAMP,YES,None,None,None
closed_date,TIMESTAMP,YES,None,None,None
complaint_type,VARCHAR,YES,None,None,None
agency,VARCHAR,YES,None,None,None
borough,VARCHAR,YES,None,None,None
status,VARCHAR,YES,None,None,None
open_data_channel_type,VARCHAR,YES,None,None,None
quality_missing_complaint_type,BOOLEAN,YES,None,None,None
quality_missing_agency,BOOLEAN,YES,None,None,None


unique_key,created_date,closed_date,complaint_type,agency,quality_missing_complaint_type,quality_missing_agency
63572326,2025-01-01 00:38:37,2025-01-01 01:14:24,Noise - Commercial,NYPD,False,False
63572331,2025-01-01 00:37:52,2025-01-01 00:50:50,Illegal Fireworks,NYPD,False,False
63572341,2025-01-01 00:59:00,2025-01-01 01:13:08,Noise - Commercial,NYPD,False,False


[('63572326',
  datetime.datetime(2025, 1, 1, 0, 38, 37),
  datetime.datetime(2025, 1, 1, 1, 14, 24),
  'Noise - Commercial',
  'NYPD',
  False,
  False),
 ('63572331',
  datetime.datetime(2025, 1, 1, 0, 37, 52),
  datetime.datetime(2025, 1, 1, 0, 50, 50),
  'Illegal Fireworks',
  'NYPD',
  False,
  False),
 ('63572341',
  datetime.datetime(2025, 1, 1, 0, 59),
  datetime.datetime(2025, 1, 1, 1, 13, 8),
  'Noise - Commercial',
  'NYPD',
  False,
  False)]

## Модель

Одна строка таблицы `service_requests` представляет одно обращение, сохранённое в staging ЛР № 3.
Естественный ключ `unique_key` идентифицирует обращение, решение о повторах принято в ЛР № 3 и повторно проверяется ниже

Выбрана одна широкая таблица с описаниями типа проблемы и агентства в каждой строке.
Для этого вопроса отдельные измерения и соединения не нужны, гранулярность сохраняется

`request_count` равен 1, единица измерения одно обращение.
Его можно суммировать по непересекающимся группам агентств, тем и периодам создания, либо использовать COUNT(*)

`resolution_hours` хранит разность дат закрытия и создания в часах.
Медиана рассчитывается по заполненным значениям каждой группы, складывать или усреднять медианы групп нельзя.
Сумма часов не является показателем выбранного вопроса

`created_date` и `closed_date` являются временем событий без добавленного часового пояса.
Дата сбора сырого снимка в manifest ЛР № 2 равна 2026-10-14, `source_ingested_on=2026-10-14` является логической датой ingestion.
Дата загрузки не заменяет время события, а закрытие может произойти позже января 2025 года

In [4]:
con.execute("""
    CREATE TEMP VIEW service_requests AS
    SELECT
        unique_key,
        created_date,
        closed_date,
        complaint_type,
        agency,
        1::INTEGER AS request_count,
        date_diff('microsecond', created_date, closed_date) / 3600000000.0
            AS resolution_hours,
        quality_missing_complaint_type,
        quality_missing_agency
    FROM input_staging
""")
show_query("SELECT * FROM service_requests ORDER BY unique_key LIMIT 3")

unique_key,created_date,closed_date,complaint_type,agency,request_count,resolution_hours,quality_missing_complaint_type,quality_missing_agency
63572326,2025-01-01 00:38:37,2025-01-01 01:14:24,Noise - Commercial,NYPD,1,0.5963888888888889,False,False
63572331,2025-01-01 00:37:52,2025-01-01 00:50:50,Illegal Fireworks,NYPD,1,0.21611111111111111,False,False
63572341,2025-01-01 00:59:00,2025-01-01 01:13:08,Noise - Commercial,NYPD,1,0.23555555555555555,False,False


[('63572326',
  datetime.datetime(2025, 1, 1, 0, 38, 37),
  datetime.datetime(2025, 1, 1, 1, 14, 24),
  'Noise - Commercial',
  'NYPD',
  1,
  0.5963888888888889,
  False,
  False),
 ('63572331',
  datetime.datetime(2025, 1, 1, 0, 37, 52),
  datetime.datetime(2025, 1, 1, 0, 50, 50),
  'Illegal Fireworks',
  'NYPD',
  1,
  0.21611111111111111,
  False,
  False),
 ('63572341',
  datetime.datetime(2025, 1, 1, 0, 59),
  datetime.datetime(2025, 1, 1, 1, 13, 8),
  'Noise - Commercial',
  'NYPD',
  1,
  0.23555555555555555,
  False,
  False)]

## Запись и повторное чтение

Поля перечислены явно, фильтрации, удаления повторов и JOIN нет.
Значения и оба флага качества переносятся без исправлений.
NULL даты закрытия даёт NULL времени до закрытия

Таблица записывается в существующий бакет mart.
Повторное выполнение перезаписывает тот же ключ, логическая дата взята из входного пути staging

In [5]:
def sql_literal(value):
    return "'" + value.replace("'", "''") + "'"


con.execute(f"""
    COPY (SELECT * FROM service_requests ORDER BY unique_key)
    TO {sql_literal(MART_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.read_parquet(MART_PATH, hive_partitioning=False).create_view("saved_mart")
print(MART_PATH)

s3://mart/nyc311_manhattan/source_ingested_on=2026-10-14/service_requests.parquet


## Проверки сохранённой витрины

Все проверки ниже выполняются на файле, повторно прочитанном из mart через S3

### Количество и схема

Число строк должно совпасть со staging, дополнительные исключения отсутствуют.
Имена и типы проверяются для каждого из девяти выходных полей

In [6]:
mart_rows = con.execute("SELECT count(*) FROM saved_mart").fetchone()[0]
show_rows(["staging", "mart", "Исключено"], [[staging_rows, mart_rows, staging_rows - mart_rows]])
assert mart_rows == staging_rows
expected_schema = {
    "unique_key": "VARCHAR",
    "created_date": "TIMESTAMP",
    "closed_date": "TIMESTAMP",
    "complaint_type": "VARCHAR",
    "agency": "VARCHAR",
    "request_count": "INTEGER",
    "resolution_hours": "DOUBLE",
    "quality_missing_complaint_type": "BOOLEAN",
    "quality_missing_agency": "BOOLEAN",
}
actual_schema = {row[0]: row[1] for row in con.execute("DESCRIBE saved_mart").fetchall()}
show_query("DESCRIBE saved_mart")
assert list(actual_schema.items()) == list(expected_schema.items())

staging,mart,Исключено
61304,61304,0


column_name,column_type,null,key,default,extra
unique_key,VARCHAR,YES,None,None,None
created_date,TIMESTAMP,YES,None,None,None
closed_date,TIMESTAMP,YES,None,None,None
complaint_type,VARCHAR,YES,None,None,None
agency,VARCHAR,YES,None,None,None
request_count,INTEGER,YES,None,None,None
resolution_hours,DOUBLE,YES,None,None,None
quality_missing_complaint_type,BOOLEAN,YES,None,None,None
quality_missing_agency,BOOLEAN,YES,None,None,None


### Ключ и обязательные значения

Пропуски и повторы ключа проверяются отдельно.
Дата создания обязательна для определения периода обращения.
Пропуски типа проблемы и агентства допускаются только с соответствующими флагами, решение ЛР № 3 сохраняется

In [7]:
missing_keys = show_query("""
    SELECT count(*) AS missing_keys FROM saved_mart
    WHERE unique_key IS NULL OR trim(unique_key) = ''
""")[0][0]
duplicate_keys = show_query("""
    SELECT unique_key, count(*) AS rows_per_key FROM saved_mart
    WHERE unique_key IS NOT NULL
    GROUP BY unique_key HAVING count(*) > 1
    ORDER BY unique_key
""")
mandatory_checks = show_query(
    """
    SELECT count(*) FILTER (WHERE created_date IS NULL) AS missing_created_date,
        count(*) FILTER (
            WHERE created_date < CAST(? AS TIMESTAMP)
               OR created_date >= CAST(? AS TIMESTAMP)
        ) AS outside_period,
        count(*) FILTER (
            WHERE quality_missing_complaint_type IS DISTINCT FROM (complaint_type IS NULL)
               OR quality_missing_agency IS DISTINCT FROM (agency IS NULL)
        ) AS inconsistent_flags
    FROM saved_mart
""",
    [PERIOD_START, PERIOD_END],
)[0]
assert missing_keys == 0 and not duplicate_keys
assert mandatory_checks == (0, 0, 0)

missing_keys
0


unique_key,rows_per_key


missing_created_date,outside_period,inconsistent_flags
0,0,0


### Меры и вычисленные значения

Счётчик должен быть заполнен и равен 1 в каждой строке.
Пропуск времени до закрытия допустим только при отсутствующей дате закрытия и сверяется со staging.
Время не может быть отрицательным или бесконечным, верхняя граница не придумывается.
Минимум и максимум сравниваются с разностью исходных дат

In [8]:
measure_checks = show_query("""
    SELECT
        count(*) FILTER (WHERE request_count IS NULL) AS missing_request_count,
        min(request_count) AS min_request_count,
        max(request_count) AS max_request_count,
        sum(request_count) AS total_requests,
        count(*) FILTER (WHERE resolution_hours IS NULL) AS missing_resolution_hours,
        min(resolution_hours) AS min_resolution_hours,
        max(resolution_hours) AS max_resolution_hours,
        count(*) FILTER (
            WHERE resolution_hours < 0 OR NOT isfinite(resolution_hours)
        ) AS invalid_resolution_hours,
        count(*) FILTER (
            WHERE resolution_hours IS DISTINCT FROM
                date_diff('microsecond', created_date, closed_date) / 3600000000.0
        ) AS incorrect_calculation
    FROM saved_mart
""")[0]
input_measures = show_query("""
    SELECT count(*) FILTER (WHERE closed_date IS NULL) AS missing_closed_date,
        min(date_diff('microsecond', created_date, closed_date) / 3600000000.0)
            AS expected_min_hours,
        max(date_diff('microsecond', created_date, closed_date) / 3600000000.0)
            AS expected_max_hours
    FROM input_staging
""")[0]
assert measure_checks[0] == 0
assert measure_checks[1:3] == (1, 1)
assert measure_checks[3] == staging_rows
assert measure_checks[4] == input_measures[0]
assert measure_checks[5:7] == input_measures[1:3]
assert measure_checks[7:] == (0, 0)

missing_request_count,min_request_count,max_request_count,total_requests,missing_resolution_hours,min_resolution_hours,max_resolution_hours,invalid_resolution_hours,incorrect_calculation
0,1,1,61304,1036,0.0,15276.065555555555,0,0


missing_closed_date,expected_min_hours,expected_max_hours
1036,0.0,15276.065555555555


### Перенос исходных значений и флагов

Сравнение в обе стороны через EXCEPT ALL проверяет все переносимые поля с учётом повторов и NULL.
Оно подтверждает перенос исходных значений вместе с флагами, включая потенциальные помеченные строки.
Число установленных признаков отдельно сверяется между staging и mart

In [9]:
transferred_columns = (
    "unique_key, created_date, closed_date, complaint_type, agency, "
    "quality_missing_complaint_type, quality_missing_agency"
)
transfer_differences = show_query(f"""
    SELECT count(*) AS transfer_differences FROM (
        (SELECT {transferred_columns} FROM input_staging
         EXCEPT ALL SELECT {transferred_columns} FROM saved_mart)
        UNION ALL
        (SELECT {transferred_columns} FROM saved_mart
         EXCEPT ALL SELECT {transferred_columns} FROM input_staging)
    )
""")[0][0]
flag_counts = show_query("""
    SELECT 'staging' AS layer,
        count(*) FILTER (WHERE quality_missing_complaint_type) AS missing_type_flags,
        count(*) FILTER (WHERE quality_missing_agency) AS missing_agency_flags
    FROM input_staging
    UNION ALL
    SELECT 'mart',
        count(*) FILTER (WHERE quality_missing_complaint_type),
        count(*) FILTER (WHERE quality_missing_agency)
    FROM saved_mart
""")
assert transfer_differences == 0
assert flag_counts[0][1:] == flag_counts[1][1:]

transfer_differences
0


layer,missing_type_flags,missing_agency_flags
staging,0,0
mart,0,0


### Пригодность для выбранного вопроса

Короткая группировка по агентству и типу проблемы проверяет доступность выбранных срезов и аддитивность счётчика.
Результат ограничен пятью группами и не является полным аналитическим ответом на вопрос Project Brief

In [10]:
show_query("""
    SELECT agency, complaint_type, count(*) AS rows_in_group,
        sum(request_count) AS requests_in_group
    FROM saved_mart
    GROUP BY agency, complaint_type
    ORDER BY requests_in_group DESC, agency, complaint_type
    LIMIT 5
""")
input_after = fingerprint(STAGING_PATH)
assert input_before == input_after
output_fingerprint = fingerprint(MART_PATH)
run_summary = {
    "staging_rows": staging_rows,
    "mart_rows": mart_rows,
    "excluded_rows": 0,
    "output_fields": len(actual_schema),
    "missing_keys": missing_keys,
    "duplicate_key_groups": len(duplicate_keys),
    "transfer_differences": transfer_differences,
    "null_resolution_hours": int(measure_checks[4]),
    "min_resolution_hours": measure_checks[5],
    "max_resolution_hours": measure_checks[6],
    "quality_flags": list(flag_counts[1][1:]),
    "staging_unchanged": input_before == input_after,
    "mart_sha256": output_fingerprint["sha256"],
}
print(run_summary)
con.close()

agency,complaint_type,rows_in_group,requests_in_group
HPD,HEAT/HOT WATER,15538,15538
NYPD,Noise - Residential,5666,5666
NYPD,Illegal Parking,4903,4903
HPD,UNSANITARY CONDITION,1853,1853
NYPD,Noise - Commercial,1843,1843


{'staging_rows': 61304, 'mart_rows': 61304, 'excluded_rows': 0, 'output_fields': 9, 'missing_keys': 0, 'duplicate_key_groups': 0, 'transfer_differences': 0, 'null_resolution_hours': 1036, 'min_resolution_hours': 0.0, 'max_resolution_hours': 15276.065555555555, 'quality_flags': [0, 0], 'staging_unchanged': True, 'mart_sha256': '71aeeef88c3d6c3c2102a4d1ef9700b94c40e14ae312bc20abd9cf52ef311048'}


## Итог и ограничения

Гранулярность и число строк сохранены, дополнительные фильтры не применялись.
Витрина позволяет считать обращения по типам проблем и агентствам и рассчитывать медиану наблюдаемого времени до закрытия по заполненным значениям

Подсчёты относятся к допустимым записям staging, 27 исключений ЛР № 3 в витрину не возвращаются.
Обращения без даты закрытия входят в число обращений, но не в медиану времени.
Источник является динамическим снимком, закрытие записи не доказывает фактическое решение проблемы

Модель, происхождение всех полей и краткий итог проверок описаны в `docs/mart.md`